In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from machine_learning.transform_dataset_ml import transform_dataset
from machine_learning.create_ml_dataset import create_dataset_from_all_file
import numpy as np
import pandas as pd
from mlflow.models.signature import infer_signature
import mlflow.sklearn
import mlflow

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from typing import Dict, Any



from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer

import xgboost as xgb
from sklearn.base import BaseEstimator
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns 




In [2]:
ml_dataset = create_dataset_from_all_file()

Machine learning dataset create


In [3]:
ml_dataset.shape

(239, 34)

In [4]:
ml_dataset.columns

Index(['company_name', 'date_bourse', 'open_price', 'close_price', 'low_price',
       'high_price', 'finnhubIndustry', 'mean_sell', 'mean_strongBuy',
       'mean_strongSell', 'insider_sentiment', 'company_posit_dist_percet',
       'company_negat_dist_percet', 'country_posit_dist_percet',
       'country_negat_dist_percet', 'analyst_sentiment_distribution',
       'analyst_price_target', 'volume_action', 'this_month_most_direction',
       'last_month_most_direction', 'year_to_date_direction',
       'one_year_rolling_period_most_direction',
       'five_year_rolling_period_most_direction', 'this_month_trend',
       'last_month_trend', 'year_to_date_trend', 'one_year_rolling_trend',
       'five_year_rollingtrend', 'this_month_volume', 'last_month_volume',
       'year_to_date_volume', 'one_year_rolling_volume',
       'five_year_rolling_volume', 'analyst_recommendation_of_the_month'],
      dtype='object')

# trasnform dataset

In [6]:
encode_dataset, X, y, label_encoder = transform_dataset(ml_dataset.copy())

### descriptive info dataset

In [8]:
encode_dataset.shape

(239, 41)

In [10]:
# encode_dataset.describe()

#### hist

In [9]:
# encode_dataset.hist(figsize=(8*4, 6*4))

#### plot

In [8]:
# encode_dataset.plot(kind='box', subplots=True, layout=(6,7), sharex=False, sharey=False, figsize=(8*4, 6*4))

## train test_split

- Cas multi-classes → toujours utiliser stratify=y dans train_test_split pour que la répartition des classes soit conservée.
- test_size à 0.25 car dataset petit

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42,
    stratify=y 
)

In [13]:
# from collections import Counter
# Counter(y_test)
# Counter(y_train)

#### dataset petit et déséquilibré (3 classes).
- scikit-learn (LR / SVC / RF) : class_weight="balanced".
- XGBoost (multi-classes) : pas de class_weight natif → passe sample_weight à fit() (poids = 1 / fréquence de classe).

In [10]:
models = {}
params_models_dict = {}

# LogisticRegression
models["LogisticRegression"] = LogisticRegression(multi_class="multinomial", solver="lbfgs", class_weight="balanced", max_iter=2000, n_jobs=None)
params_models_dict["LogisticRegression"] = logreg_grid = {
    "clf__penalty": ["l2"],                  # L2 robuste
    "clf__C": [0.1, 0.5, 1.0, 2.0],         # régularisation légère → moyenne
    "clf__solver": ["lbfgs"],               # multinomial OK
    "clf__multi_class": ["multinomial"],
    "clf__class_weight": ["balanced"],
    "clf__max_iter": [1000, 2000]
}


# SVC
models["SVC"] = SVC(kernel="rbf", class_weight="balanced", probability=False)
params_models_dict["SVC"] = svc_grid = {
    "clf__kernel": ["rbf"],
    "clf__C": [0.5, 1.0, 2.0, 5.0],         # gamme courte
    "clf__gamma": ["scale", 0.1, 0.01],     # RBF width
    "clf__class_weight": ["balanced"]
}


# RandomForestClassifier
models["RandomForestClassifier"] = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
params_models_dict["RandomForestClassifier"] = rf_grid = {
    "clf__n_estimators": [200, 400, 600],   # petit → 200–600 suffit
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", 0.5],     # sous-échantillonnage des features
    "clf__class_weight": ["balanced_subsample"],
    "clf__random_state": [42]
}


# xgb
models["xgb"] = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist")
params_models_dict["xgb"] = xgb_grid = {
    "clf__objective": ["multi:softprob"],
    "clf__num_class": [3],                 # 3 classes
    "clf__n_estimators": [200, 400],       # peu d’arbres + early stopping
    "clf__max_depth": [3, 5],
    "clf__learning_rate": [0.03, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
    "clf__reg_lambda": [1.0, 2.0],         # L2
    "clf__min_child_weight": [1, 5],
    "clf__tree_method": ["hist"],
    "clf__random_state": [42]
}

In [ ]:
# def needs_scaling(model_name: str) -> bool:
#     """Retourne True si un StandardScaler est requis (LR/SVC)."""
#     return model_name in {"LogisticRegression", "SVC"}

# def run_multi_gridsearch(
#     X_train: pd.DataFrame,
#     y_train: pd.Series,
#     X_test: pd.DataFrame,
#     y_test: pd.Series,
#     models: Dict[str, Any],
#     params_models_dict: Dict[str, Dict[str, Any]],
#     random_state: int = 42,
#     n_splits: int = 5,
#     scoring: Dict[str, str] | str = "f1_macro",  # accepte str OU dict
#     refit: str | bool = "f1_macro",              # si dict → clé à utiliser, sinon True/False
#     use_sample_weight_for_xgb: bool = True
# ) -> Dict[str, Any]:
#     """
#     Lance un GridSearch pour chaque modèle :
#       - ajoute automatiquement StandardScaler pour LR/SVC,
#       - 'passthrough' pour RF/XGB,
#       - CV stratifiée,
#       - renvoie un leaderboard + les meilleurs objets par modèle.
#     """
#     cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
#     sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

#     rows = []
#     best_per_model = {}

#     for name, est in models.items():
#         prep = StandardScaler() if needs_scaling(name) else "passthrough"
#         pipe = Pipeline([("prep", prep), ("clf", est)])

#         grid = params_models_dict[name]

#         gs = GridSearchCV(
#             estimator=pipe,
#             param_grid=grid,
#             scoring=scoring,       # peut être "f1_macro" ou {"acc":"accuracy","f1_macro":"f1_macro"}
#             cv=cv,
#             n_jobs=-1,
#             refit=refit,           # si scoring est dict → ex. "f1_macro"
#             verbose=0,
#             return_train_score=False
#         )

#         fit_kwargs = {}
#         if use_sample_weight_for_xgb and name.lower().startswith("xgb"):
#             fit_kwargs = {"clf__sample_weight": sample_weight}

#         gs.fit(X_train, y_train, **fit_kwargs)

#         # Récup CV scores (selon le type de scoring)
#         if isinstance(scoring, dict):
#             # on lit les deux métriques si présentes
#             mean_acc = gs.cv_results_.get("mean_test_acc", [np.nan])[gs.best_index_]
#             mean_f1  = gs.cv_results_.get("mean_test_f1_macro", [np.nan])[gs.best_index_]
#             cv_key_for_leaderboard = refit if isinstance(refit, str) else "f1_macro"
#             cv_best = gs.best_score_  # score refit
#         else:
#             # scoring est un str (ex. "f1_macro")
#             mean_acc = np.nan
#             mean_f1  = gs.best_score_ if scoring == "f1_macro" else np.nan
#             cv_key_for_leaderboard = scoring
#             cv_best = gs.best_score_

#         # Perf test
#         y_pred = gs.best_estimator_.predict(X_test)
#         test_acc = accuracy_score(y_test, y_pred)
#         test_f1  = f1_score(y_test, y_pred, average="macro")

#         best_per_model[name] = gs.best_estimator_
#         rows.append({
#             "model": name,
#             "best_params": gs.best_params_,
#             f"cv_{cv_key_for_leaderboard}": cv_best,
#             "cv_mean_acc": mean_acc,
#             "cv_mean_f1_macro": mean_f1,
#             "test_acc": test_acc,
#             "test_f1_macro": test_f1
#         })

#     leaderboard = pd.DataFrame(rows).sort_values(
#         by=[col for col in [f"cv_{refit}" if isinstance(refit, str) else f"cv_{scoring}"] if col in rows[0]],
#         ascending=False
#     ).reset_index(drop=True)

#     return {"leaderboard": leaderboard, "best_per_model": best_per_model}


In [11]:
from typing import Dict, Any, Literal
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight


def needs_scaling(model_name: str) -> bool:
    """Retourne True si un StandardScaler est requis (LR/SVC)."""
    return model_name in {"LogisticRegression", "SVC"}


def run_multi_gridsearch(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    models: Dict[str, Any],
    params_models_dict: Dict[str, Dict[str, Any]],
    *,
    random_state: int = 42,
    n_splits: int = 5,
    # Choix de la moyenne de référence pour le refit/tri : "macro" (égale importance) ou "weighted" (pondéré par fréquence)
    prefer_avg: Literal["macro", "weighted"] = "weighted",
    use_sample_weight_for_xgb: bool = True
) -> Dict[str, Any]:
    """
    GridSearch multi-modèles avec CV stratifiée :
      - StandardScaler auto pour LR/SVC, passthrough sinon,
      - scoring multi-métriques : accuracy, f1_macro, f1_weighted, recall_macro, recall_weighted,
      - refit/tri sur f1_{prefer_avg} (par défaut f1_macro),
      - log des métriques TEST : acc, f1_macro/weighted, recall_macro/weighted.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

    # Scoring multi-métriques (CV)
    scoring = {
        "acc": "accuracy",
        "f1_macro": "f1_macro",
        "f1_weighted": "f1_weighted",
        "recall_macro": "recall_macro",
        "recall_weighted": "recall_weighted",
    }
    refit_key = f"f1_{prefer_avg}"  # "f1_macro" (par défaut) ou "f1_weighted"

    rows = []
    best_per_model = {}

    for name, est in models.items():
        prep = StandardScaler() if needs_scaling(name) else "passthrough"
        pipe = Pipeline([("prep", prep), ("clf", est)])
        grid = params_models_dict[name]

        gs = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            refit=refit_key,     # on retient le meilleur sur f1_macro ou f1_weighted
            verbose=0,
            return_train_score=False,
        )

        fit_kwargs = {}
        if use_sample_weight_for_xgb and name.lower().startswith("xgb"):
            fit_kwargs = {"clf__sample_weight": sample_weight}

        gs.fit(X_train, y_train, **fit_kwargs)

        # Récup meilleures moyennes CV
        idx = gs.best_index_
        get_cv = lambda key: gs.cv_results_.get(f"mean_test_{key}", [np.nan])[idx]

        cv_best_refit        = gs.best_score_              # f1_{prefer_avg}
        cv_mean_acc          = get_cv("acc")
        cv_mean_f1_macro     = get_cv("f1_macro")
        cv_mean_f1_weighted  = get_cv("f1_weighted")
        cv_mean_rec_macro    = get_cv("recall_macro")
        cv_mean_rec_weighted = get_cv("recall_weighted")

        # Scores TEST
        y_pred = gs.best_estimator_.predict(X_test)
        test_acc            = accuracy_score(y_test, y_pred)
        test_f1_macro       = f1_score(y_test, y_pred, average="macro")
        test_f1_weighted    = f1_score(y_test, y_pred, average="weighted")
        test_recall_macro   = recall_score(y_test, y_pred, average="macro")
        test_recall_weighted= recall_score(y_test, y_pred, average="weighted")

        best_per_model[name] = gs.best_estimator_
        rows.append({
            "model": name,
            "best_params": gs.best_params_,
            f"cv_{refit_key}": cv_best_refit,
            "cv_mean_acc": cv_mean_acc,
            "cv_mean_f1_macro": cv_mean_f1_macro,
            "cv_mean_f1_weighted": cv_mean_f1_weighted,
            "cv_mean_recall_macro": cv_mean_rec_macro,
            "cv_mean_recall_weighted": cv_mean_rec_weighted,
            "test_acc": test_acc,
            "test_f1_macro": test_f1_macro,
            "test_f1_weighted": test_f1_weighted,
            "test_recall_macro": test_recall_macro,
            "test_recall_weighted": test_recall_weighted,
        })

    sort_col = f"cv_{refit_key}"
    leaderboard = pd.DataFrame(rows).sort_values(by=sort_col, ascending=False).reset_index(drop=True)
    return {"leaderboard": leaderboard, "best_per_model": best_per_model}


In [16]:
# scores = {"acc": "accuracy", "f1_macro": "f1_macro"}

# # cross validation
# min_cv_folds=5
# min_class_count = pd.Series(y_train).value_counts().min()
# cv_folds = max(3, min(min_cv_folds, min_class_count))

# out = run_multi_gridsearch(
#     X_train, y_train, X_test, y_test,
#     models, params_models_dict,
#     n_splits=cv_folds,
#     scoring=scores, refit="f1_macro"
# )

In [ ]:
# out["leaderboard"]

NameError: name 'out' is not defined

In [ ]:
# print(out["best_per_model"])

---

# ML flows

In [ ]:
# mlflow.set_tracking_uri("http://127.0.0.1:5000")  # ou un chemin local: "file:./mlruns"
# mlflow.set_experiment("tradehelper_classif_v1")

# MLFlow ce que j'enregistre et auquel je peux avoir accès 
L'experimenattion en elel même est sauvegardé avec un dataframe de récap qui contient :
- le nom du model, 
- les meilleurs parametres trouvées
- les résultat en cross_validation f1_macro
- la moyenne de tout les réusltat accuracy en cross_validation moyenne
- la moyenne de tout les réusltat f1_score en cross_validation moyenne
- le test_accracy
- test_f1_macro

Pour chaque model tester mais uniquement pour les meillueurs parametres trouver avec le grid search on enregistre :
- plot seaborn de ma confusion matrix
- le classification report 
- le test_accuracy
- le test_f1_macro
- les parametres associé au model
- le nom du model
- on sauvegarde le model en .pkl si on souhaite le retelecharger après 
  - (En gros, ça veut dire que tu sauvegardes le modèle entraîné + son environnement dans l’historique MLflow, pour assurer la traçabilité et pouvoir le rejouer plus tard.)


In [16]:
def log_with_mlflow(
        set_experiment_name,
        X_train, y_train, X_test, y_test,
        models, params_models_dict,
        n_splits, class_names
):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment(set_experiment_name)
    with mlflow.start_run(run_name="multi_models_gridsearch") as parent_run:
        # 1) Lancer ton gridsearch multi-modèles

        out = run_multi_gridsearch(
            X_train, y_train, X_test, y_test,
            models, params_models_dict,
            n_splits=n_splits,
            # scoring={"acc":"accuracy", "f1_macro":"f1_macro"},
            # refit="f1_macro"
        )

        # 2) Logger le leaderboard comme artefact CSV
        leaderboard = out["leaderboard"]
        leaderboard.to_csv("leaderboard.csv", index=False)
        mlflow.log_artifact("leaderboard.csv")

        # 3) Pour chaque meilleur modèle, créer un run enfant et tout logger
        for name, best_model in out["best_per_model"].items():
            with mlflow.start_run(run_name=f"best_{name}", nested=True):
                # Log params du meilleur pipeline
                mlflow.log_params(best_model.get_params(deep=True))

                # Scores test
                y_pred = best_model.predict(X_test)
                cm = confusion_matrix(y_test, y_pred)
                report_str = classification_report(y_test, y_pred, target_names=class_names)

                # Log metrics (quelques-unes)
                from sklearn.metrics import f1_score, accuracy_score
                mlflow.log_metric("test_f1_macro", f1_score(y_test, y_pred, average="macro"))
                mlflow.log_metric("test_acc", accuracy_score(y_test, y_pred))

                # Confusion matrix en image
                fig, ax = plt.subplots(figsize=(4,3))
                sns.heatmap(cm, annot=True, fmt="d", cbar=False, ax=ax,
                            xticklabels=class_names, yticklabels=class_names)
                ax.set_xlabel("Pred"); ax.set_ylabel("True"); ax.set_title(f"CM - {name}")
                fig.savefig("cm.png", bbox_inches="tight")
                plt.close(fig)
                mlflow.log_artifact("cm.png")

                # Rapport de classification en txt
                with open("classification_report.txt", "w") as f:
                    f.write(report_str)
                mlflow.log_artifact("classification_report.txt")

                # Log du modèle (pickle + conda/env)
                model_name = "model_" + name
                input_example = X_train.iloc[:5].copy()
                signature = infer_signature(input_example, y_pred)
                # mlflow.sklearn.log_model(
                #     best_model, name=model_name, 
                #     artifact_path="model_weight",
                #     signature=signature, input_example=input_example)#artifact_path="model") # mettre le nom du model ici 
                
                artifact_path = "model_weight_" + name
                mlflow.sklearn.log_model(
                    sk_model=best_model,
                    artifact_path="models",
                    signature=signature,
                    input_example=input_example
                )

        # Optionnel: tag général
        mlflow.set_tag("project", "TradeHelper")


In [17]:
# cross validation
min_cv_folds=5
min_class_count = pd.Series(y_train).value_counts().min()
cv_folds = max(3, min(min_cv_folds, min_class_count))

experimentation_name = "tradehelper_classif_v5"

log_with_mlflow(
    set_experiment_name=experimentation_name,
    X_train= X_train,
    y_train= y_train,
    X_test= X_test,
    y_test= y_test,
    models= models,
    params_models_dict= params_models_dict,
    n_splits=cv_folds,
    class_names=[x for x in set(label_encoder.inverse_transform(y))]
)

c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\EnvBourseMachineLearning\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\EnvBourseMachineLearning\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these

🏃 View run best_LogisticRegression at: http://127.0.0.1:5000/#/experiments/879760805602818584/runs/601edac06d364dcda0c1afa8f9a2a2c5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/879760805602818584


c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\EnvBourseMachineLearning\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/09/06 13:04:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run best_SVC at: http://127.0.0.1:5000/#/experiments/879760805602818584/runs/182995a8ee424e1aa1db084d853ffc0e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/879760805602818584


c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\EnvBourseMachineLearning\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/09/06 13:04:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run best_RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/879760805602818584/runs/f27c89c0fde949b8947fb365ee28122b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/879760805602818584


c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\EnvBourseMachineLearning\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/09/06 13:04:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run best_xgb at: http://127.0.0.1:5000/#/experiments/879760805602818584/runs/9d8fe6ab3efe4161a24609e3a181e575
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/879760805602818584
🏃 View run multi_models_gridsearch at: http://127.0.0.1:5000/#/experiments/879760805602818584/runs/9d91f63f08b84ebfbab717dcc97f4f73
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/879760805602818584


In [ ]:
# load the model that have the best _pred

In [ ]:
# run_id = mlflow.last_active_run().info.run_id

# # Chemin complet vers le modèle
# model_uri = f"runs:/{run_id}/model_rf"

In [ ]:
# 769400862886965147 => experimentation ID
# model ID ==> m-34e6065deadd43beaddd155f5f969250
model_path = r"Projet_master\mlartifacts\769400862886965147\models\m-34e6065deadd43beaddd155f5f969250\artifacts\model.pkl"


In [ ]:
current_path = Path.cwd()
parent_dir = current_path.parent.absolute().parent.absolute()
full_model_path = parent_dir/model_path

try:
    with open(full_model_path, 'rb') as file:
        model = pickle.load(file)
    if isinstance(model, BaseEstimator):  # modèle sklearn
        print("✅ Modèle Sklearn chargé :", type(model))
        # Exemple : prédiction
        # y_pred = model.predict(X_test)

    elif isinstance(model, xgb.XGBModel):  # XGBClassifier ou XGBRegressor
        print("✅ Modèle XGBoost chargé :", type(model))
        # Exemple : prédiction
        # y_pred = model.predict(X_test)

    else:
        print("⚠️ Ce fichier ne contient pas un modèle reconnu :", type(model))
except FileNotFoundError:
    print(f"File not found: {file_path}")
except pickle.UnpicklingError:
    print("Error: The file content is not a valid pickle format.")
except EOFError:
    print("Error: The file is incomplete or corrupted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

✅ Modèle Sklearn chargé : <class 'sklearn.pipeline.Pipeline'>


In [68]:
new_data = create_dataset_from_all_file()

Machine learning dataset create


In [84]:
encode_dataset, X, y, label_encoder = transform_dataset(new_data)

In [92]:
filter_company = X["company_name_anf"]==1
filter_most_recent_day = X["date_weight"]==7

In [95]:
abercombie_last_day = X[filter_company & filter_most_recent_day]

In [96]:
y_pred = model.predict(abercombie_last_day)

In [97]:
y_pred

array([2])